# Notebook 3 – Train, Validation & Test

## Introduction

Training, validation, and testing are important stages in machine learning. They help us train models, select the best model, and evaluate performance on unseen data.

In this notebook, we will learn how to split datasets into training, validation, and testing sets. We will also explore cross-validation techniques and understand data leakage, overfitting, and underfitting.

**Dataset:** Car Price Prediction

**Target Variable:** Price

**Objective:** To understand different data-splitting techniques and evaluate machine learning models reliably.
_______________________________________________________________________________________________________________

## Import Libraries

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import (
    train_test_split,
    KFold,
    StratifiedKFold,
    cross_val_score
)

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("Libraries imported successfully!")

Libraries imported successfully!


# Load the Car Dataset.

In [3]:
car_df = pd.read_csv(r"datasets/car_prediction/car_price_prediction_.csv")

print("Dataset loaded successfully!")
print("Dataset Shape:", car_df.shape)

display(car_df.head())

Dataset loaded successfully!
Dataset Shape: (2500, 10)


,Car ID,Brand,Year,Engine Size,Fuel Type,Transmission,Mileage,Condition,Price,Model
0,1,Tesla,2016,2.3,Petrol,Manual,114832,New,26613.92,Model X
1,2,BMW,2018,4.4,Electric,Manual,143190,Used,14679.61,5 Series
2,3,Audi,2013,4.5,Electric,Manual,181601,New,44402.61,A4
3,4,Tesla,2011,4.1,Diesel,Automatic,68682,New,86374.33,Model Y
4,5,Ford,2009,2.6,Diesel,Manual,223009,Like New,73577.10,Mustang


# Separate Features and Target

In [4]:
X = car_df.drop(columns=["Price", "Car ID"])
y = car_df["Price"]

print("Features Shape:", X.shape)
print("Target Shape:", y.shape)

Features Shape: (2500, 8)
Target Shape: (2500,)


# Training Dataset

The training dataset is used to teach the machine learning model. The model learns patterns and relationships between input features and the target variable.

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42
)

print("Training Dataset Shape:", X_train.shape)
print("Testing Dataset Shape:", X_test.shape)

Training Dataset Shape: (2000, 8)
Testing Dataset Shape: (500, 8)


Validation Dataset


The validation dataset is used during model development to evaluate model performance and tune hyperparameters. It helps us select the best model before final testing.

Test Dataset


The test dataset contains unseen records. It is used to evaluate the final model and understand how well it performs on new data.

# Train/Test Split

  Train/Test Split divides the dataset into two parts:

    - Training data: Used to train the model.
    - Testing data: Used for final evaluation.

In [6]:
print("Training Records:", len(X_train))
print("Testing Records:", len(X_test))

print("Training Percentage:", len(X_train) / len(X) * 100)
print("Testing Percentage:", len(X_test) / len(X) * 100)

Training Records: 2000
Testing Records: 500
Training Percentage: 80.0
Testing Percentage: 20.0


# Train/Validation/Test Split

    The dataset is divided into three parts:
    - 70% Training data
    - 15% Validation data
    - 15% Testing data

    Training data is used to train the model, validation data is used to tune it, and testing data is used for final evaluation.

In [7]:
X_train_val, X_test_final, y_train_val, y_test_final = train_test_split(
    X, y,
    test_size=0.15,
    random_state=42
)

X_train_final, X_val, y_train_final, y_val = train_test_split(
    X_train_val,
    y_train_val,
    test_size=0.1765,
    random_state=42
)

print("Training Data:", X_train_final.shape)
print("Validation Data:", X_val.shape)
print("Testing Data:", X_test_final.shape)

Training Data: (1749, 8)
Validation Data: (376, 8)
Testing Data: (375, 8)


# Cross Validation

Cross-validation is a technique used to evaluate a machine learning model by dividing the training dataset into multiple parts.

The model is trained and evaluated several times using different portions of the data. It provides a more reliable estimate of model performance.

# K-Fold Cross Validation

K-Fold Cross Validation divides the dataset into K equal-sized folds.

In each iteration:
- One fold is used for validation.
- The remaining folds are used for training.
- This process repeats until every fold has been used for validation.

In [8]:
X_cv = pd.get_dummies(X_train_final).fillna(0)

kfold = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

cv_scores = cross_val_score(
    model,
    X_cv,
    y_train_final,
    cv=kfold,
    scoring="r2"
)

print("K-Fold Cross Validation Scores:", cv_scores)
print("Average CV Score:", cv_scores.mean())

K-Fold Cross Validation Scores: [-0.02851526 -0.06558813 -0.04021192  0.00064524 -0.11320016]
Average CV Score: -0.049374043161482015


# Stratified K-Fold

Stratified K-Fold maintains the proportion of different classes in each fold. It is mainly used for classification problems.

Since Car Price Prediction is a regression problem, we divide continuous car prices into groups (bins). We can then use those groups to demonstrate stratified splitting.

In [9]:
price_groups = pd.qcut(
    y_train_final,
    q=5,
    labels=False,
    duplicates="drop"
)

stratified_kfold = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

for train_index, val_index in stratified_kfold.split(
    X_train_final,
    price_groups
):
    print("Training Fold:", len(train_index))
    print("Validation Fold:", len(val_index))

Training Fold: 1399
Validation Fold: 350
Training Fold: 1399
Validation Fold: 350
Training Fold: 1399
Validation Fold: 350
Training Fold: 1399
Validation Fold: 350
Training Fold: 1400
Validation Fold: 349


# Random State

 Random state controls the randomness involved in splitting data and training models.

 Using the same random state produces reproducible results. For example, random_state=42 helps us obtain the same data split whenever we execute the code with unchanged data.

In [10]:
X1_train, X1_test, y1_train, y1_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

X2_train, X2_test, y2_train, y2_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Same Training Split:", X1_train.equals(X2_train))
print("Same Testing Split:", X1_test.equals(X2_test))

Same Training Split: True
Same Testing Split: True


# Data Leakage

    Data leakage occurs when information from outside the training dataset is accidentally used during model training.

    Examples:
    - Using test data during model training.
    - Performing preprocessing on the entire dataset before splitting.
    - Using future information to predict past outcomes.

    Data leakage can produce misleadingly good model performance.

In [11]:
from sklearn.preprocessing import StandardScaler

X_numeric = X.select_dtypes(include=np.number).fillna(0)

X_train_leak, X_test_leak, y_train_leak, y_test_leak = train_test_split(
    X_numeric, y,
    test_size=0.20,
    random_state=42
)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_leak)
X_test_scaled = scaler.transform(X_test_leak)

print("Data Leakage Demonstration Completed!")
print("Training Data Shape:", X_train_scaled.shape)
print("Testing Data Shape:", X_test_scaled.shape)

Data Leakage Demonstration Completed!
Training Data Shape: (2000, 3)
Testing Data Shape: (500, 3)


# Overfitting

    Overfitting occurs when a machine learning model learns the training data too closely, including noise and unnecessary details.

    Characteristics:
    - Very high training performance.
    - Poor validation or testing performance.
    - Poor predictions on unseen data.
    
    Ways to reduce overfitting:
    - Use cross-validation.
    - Reduce model complexity.
    - Apply regularization.
    - Collect more training data.

In [12]:
from sklearn.tree import DecisionTreeRegressor

overfit_model = DecisionTreeRegressor(random_state=42)

X_overfit = pd.get_dummies(X_train_final).fillna(0)
X_val_overfit = pd.get_dummies(X_val).reindex(
    columns=X_overfit.columns, fill_value=0
)

overfit_model.fit(X_overfit, y_train_final)

train_score = overfit_model.score(X_overfit, y_train_final)
val_score = overfit_model.score(X_val_overfit, y_val)

print("Training R2 Score:", train_score)
print("Validation R2 Score:", val_score)

Training R2 Score: 1.0
Validation R2 Score: -1.069247269495047


### Overfitting Result

The Decision Tree model achieved an R² score of 1.0 on training data and -1.069 on validation data.

This indicates overfitting because the model learned the training data very closely but performed poorly on unseen validation data.

**Conclusion:** The model does not generalize well to new data.
___________________________________________________________________________________________________________________

# Underfitting

    Underfitting occurs when a model is too simple to learn the important patterns in the dataset.

    Characteristics:
    - Poor training performance.
    - Poor validation performance.
    - The model cannot capture important relationships.

    Ways to reduce underfitting:
    - Use a more complex model.
    - Add useful features.
    - Improve feature engineering.
    - Reduce excessive regularization.

In [13]:
from sklearn.tree import DecisionTreeRegressor

underfit_model = DecisionTreeRegressor(
    max_depth=1,
    random_state=42
)

underfit_model.fit(X_overfit, y_train_final)

train_score_underfit = underfit_model.score(
    X_overfit, y_train_final
)

val_score_underfit = underfit_model.score(
    X_val_overfit, y_val
)

print("Training R2 Score:", train_score_underfit)
print("Validation R2 Score:", val_score_underfit)

Training R2 Score: 0.0046549198719735285
Validation R2 Score: -0.009660527847358491


### Underfitting Result

The Decision Tree model achieved a training R² score of 0.00465 and a validation R² score of -0.00966.

Both scores are very low, indicating that the model is too simple to learn the important patterns in the dataset.

**Conclusion:** The model is underfitting and needs more complexity to improve its performance.
___________________________________________________________________________________________________________________

Why Should We Not Use Test Data for Model Tuning?

    The test dataset should not be used for model tuning because it must represent completely unseen data.

    If we repeatedly use test data to adjust model parameters, the model indirectly learns information about the test dataset. This makes the final evaluation unreliable.

    Therefore:
    
    - Training data is used to train the model.
    - Validation data is used to tune and select the model.
    - Test data is used only for final evaluation.
    _______________________________________________________________________________________________________________

Conclusion

    In this notebook, we learned about training, validation, and testing datasets. We explored Train/Test Split, Train/Validation/Test Split, Cross Validation, K-Fold, Stratified K-Fold, and Random State.

    We also understood data leakage, overfitting, underfitting, and why test data should not be used for model tuning.

    These techniques help build reliable machine learning models and evaluate their performance on unseen data.

# Final Model Evaluation

In [14]:
final_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

X_train_encoded = pd.get_dummies(X_train_final).fillna(0)
X_test_encoded = pd.get_dummies(X_test_final).reindex(
    columns=X_train_encoded.columns,
    fill_value=0
)

final_model.fit(X_train_encoded, y_train_final)

final_predictions = final_model.predict(X_test_encoded)

print("FINAL TEST RESULTS")
print("MAE:", mean_absolute_error(y_test_final, final_predictions))
print("RMSE:", np.sqrt(mean_squared_error(y_test_final, final_predictions)))
print("R2 Score:", r2_score(y_test_final, final_predictions))

FINAL TEST RESULTS
MAE: 24313.754556800002
RMSE: 28464.38848533797
R2 Score: -0.05747953043864995


Result Explanation

    - MAE: Average prediction error is approximately 24,314 price units.
    - RMSE: Indicates prediction errors are relatively high.
    - R² Score: Negative, meaning the model performs worse than simply predicting the average car price 
      on this test set.